# Weeks 4-5: California and Flagged Row Investigation

This notebook investigates the first-stage cleaned sold dataset before final cleaning decisions are applied. It reviews state issues and every data-quality flag.

In [17]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PRE_FILTER_FILE = PROCESSED_DIR / "crmls_sold_cleaned_before_ca_filter_202401_202606.csv"
pre_filter = pd.read_csv(PRE_FILTER_FILE, low_memory=False)

print("Investigation dataset shape:", pre_filter.shape)

Investigation dataset shape: (447991, 67)


## 1. Investigate State Values

Visualize the state distribution and inspect records that are not explicitly identified as California.

In [18]:
state = pre_filter["StateOrProvince"].astype("string").str.strip().str.upper()
state_counts = state.fillna("MISSING").value_counts().rename_axis("state").reset_index(name="row_count")
display(state_counts)

non_ca_mask = state.notna() & ~state.isin(["CA", "CALIFORNIA"])
missing_state_mask = state.isna() | state.eq("")

print("Explicit non-California rows:", int(non_ca_mask.sum()))
print("Missing-state rows:", int(missing_state_mask.sum()))

,state,row_count
0,CA,447961
1,AZ,10
2,OS,4
3,NV,2
4,CO,2
5,FL,2
6,ID,1
7,TX,1
8,GA,1
9,ME,1


Explicit non-California rows: 29
Missing-state rows: 1


In [19]:
display(pre_filter.loc[missing_state_mask])

,Flooring,ViewYN,PoolPrivateYN,OriginalListPrice,ListingKey,CloseDate,ClosePrice,Latitude,Longitude,UnparsedAddress,...,negative_timeline_flag,missing_coordinate_flag,zero_coordinate_flag,positive_longitude_flag,implausible_coordinate_flag,price_per_sqft,close_price_outlier_flag,living_area_outlier_flag,price_per_sqft_outlier_flag,possible_non_standard_residential_flag
434753,Wood,False,False,550000.0,1159827154,2026-06-09,550000.0,34.006183,-118.174188,2329 Connor Avenue,...,False,False,False,False,False,617.977528,False,False,False,False


In [20]:
# Create the next cleaning-stage dataset after reviewing state values.
# Keep explicit California records and the missing-state record confirmed
# by its Los Angeles County value and valid California coordinates.
plausible_ca_coordinates = (
    pre_filter["Latitude"].between(32, 42, inclusive="both")
    & pre_filter["Longitude"].between(-125, -114, inclusive="both")
)
los_angeles_county = (
    pre_filter["CountyOrParish"]
    .astype("string")
    .str.strip()
    .str.casefold()
    .eq("los angeles")
)
confirmed_missing_state_ca = (
    missing_state_mask & plausible_ca_coordinates & los_angeles_county
)
keep_ca = state.isin(["CA", "CALIFORNIA"]) | confirmed_missing_state_ca

clean_date = pre_filter.loc[keep_ca].copy()
CLEAN_DATE_FILE = PROCESSED_DIR / "crmls_sold_cleaned_analysis_ready_202401_202606.csv"
clean_date.to_csv(CLEAN_DATE_FILE, index=False)

print("Rows before state cleaning:", f"{len(pre_filter):,}")
print("Rows removed:", f"{int((~keep_ca).sum()):,}")
print("Rows in clean_date:", f"{len(clean_date):,}")
print("Saved:", CLEAN_DATE_FILE)

Rows before state cleaning: 447,991
Rows removed: 29
Rows in clean_date: 447,962
Saved: /Users/baixuezhang/Documents/IDX Exchange/real-estate-dashboard-project/data/processed/crmls_sold_cleaned_analysis_ready_202401_202606.csv


In [21]:
# Reload the saved analysis-ready CSV so every following investigation
# examines the file itself rather than the earlier in-memory dataframe.
analysis_ready = pd.read_csv(CLEAN_DATE_FILE, low_memory=False)
analysis_state = analysis_ready["StateOrProvince"].astype("string").str.strip().str.upper()

print("Analysis-ready shape:", analysis_ready.shape)
display(
    analysis_state.fillna("MISSING")
    .value_counts()
    .rename_axis("state")
    .reset_index(name="row_count")
)

Analysis-ready shape: (447962, 67)


,state,row_count
0,CA,447961
1,MISSING,1


## 2. Review Every Flagged Category

Review the quality flags already created by the cleaning script. A flag identifies a record for review and does not automatically mean that the transaction should be deleted.

In [ ]:
print(analysis_ready.columns.tolist())

In [23]:
investigation = pd.read_csv(CLEAN_DATE_FILE, low_memory=False)

date_fields = [
    "CloseDate", "PurchaseContractDate", "ListingContractDate",
    "ContractStatusChangeDate",
]
numeric_fields = [
    "ClosePrice", "ListPrice", "OriginalListPrice", "LivingArea",
    "LotSizeAcres", "BedroomsTotal", "BathroomsTotalInteger",
    "DaysOnMarket", "YearBuilt", "Latitude", "Longitude",
]

for field in date_fields:
    if field in investigation.columns:
        investigation[field] = pd.to_datetime(investigation[field], errors="coerce")

for field in numeric_fields:
    if field in investigation.columns:
        investigation[field] = pd.to_numeric(investigation[field], errors="coerce")

investigation["invalid_close_price_flag"] = investigation["ClosePrice"].le(0).fillna(False)
investigation["invalid_living_area_flag"] = investigation["LivingArea"].le(0).fillna(False)
investigation["negative_days_on_market_flag"] = investigation["DaysOnMarket"].lt(0).fillna(False)
investigation["negative_bedrooms_flag"] = investigation["BedroomsTotal"].lt(0).fillna(False)
investigation["negative_bathrooms_flag"] = investigation["BathroomsTotalInteger"].lt(0).fillna(False)

investigation["listing_after_close_flag"] = (
    investigation["ListingContractDate"].notna()
    & investigation["CloseDate"].notna()
    & investigation["ListingContractDate"].gt(investigation["CloseDate"])
)
investigation["purchase_after_close_flag"] = (
    investigation["PurchaseContractDate"].notna()
    & investigation["CloseDate"].notna()
    & investigation["PurchaseContractDate"].gt(investigation["CloseDate"])
)
purchase_before_listing = (
    investigation["PurchaseContractDate"].notna()
    & investigation["ListingContractDate"].notna()
    & investigation["PurchaseContractDate"].lt(investigation["ListingContractDate"])
)
investigation["negative_timeline_flag"] = (
    investigation["listing_after_close_flag"]
    | investigation["purchase_after_close_flag"]
    | purchase_before_listing
)

latitude = investigation["Latitude"]
longitude = investigation["Longitude"]
investigation["missing_coordinate_flag"] = latitude.isna() | longitude.isna()
investigation["zero_coordinate_flag"] = latitude.eq(0) | longitude.eq(0)
investigation["positive_longitude_flag"] = longitude.gt(0).fillna(False)
investigation["implausible_coordinate_flag"] = (
    latitude.notna() & longitude.notna()
    & (~latitude.between(32, 42) | ~longitude.between(-125, -114))
)

valid_area = investigation["LivingArea"].gt(0)
investigation["price_per_sqft"] = pd.NA
investigation.loc[valid_area, "price_per_sqft"] = (
    investigation.loc[valid_area, "ClosePrice"]
    / investigation.loc[valid_area, "LivingArea"]
)
investigation["price_per_sqft"] = pd.to_numeric(
    investigation["price_per_sqft"], errors="coerce"
)

outlier_fields = {
    "ClosePrice": "close_price_outlier_flag",
    "LivingArea": "living_area_outlier_flag",
    "price_per_sqft": "price_per_sqft_outlier_flag",
}
for field, flag in outlier_fields.items():
    threshold = investigation[field].dropna().quantile(0.99)
    investigation[flag] = investigation[field].gt(threshold).fillna(False)

investigation["possible_non_standard_residential_flag"] = (
    investigation["close_price_outlier_flag"]
    | investigation["living_area_outlier_flag"]
    | investigation["price_per_sqft_outlier_flag"]
)

print("Investigation dataset shape:", investigation.shape)

Investigation dataset shape: (447962, 67)


In [24]:
flag_columns = sorted(column for column in investigation.columns if column.endswith("_flag"))

flag_summary = pd.DataFrame({
    "flag": flag_columns,
    "flagged_count": [int(investigation[column].fillna(False).astype(bool).sum()) for column in flag_columns],
})
flag_summary["flagged_pct"] = (flag_summary["flagged_count"] / len(investigation) * 100).round(4)
display(flag_summary.sort_values("flagged_count", ascending=False))

,flag,flagged_count,flagged_pct
12,possible_non_standard_residential_flag,9798,2.1872
13,price_per_sqft_outlier_flag,4476,0.9992
5,living_area_outlier_flag,4472,0.9983
0,close_price_outlier_flag,4421,0.9869
6,missing_coordinate_flag,4375,0.9766
10,negative_timeline_flag,530,0.1183
14,purchase_after_close_flag,240,0.0536
3,invalid_living_area_flag,165,0.0368
1,implausible_coordinate_flag,85,0.0190
4,listing_after_close_flag,68,0.0152


In [25]:
review_columns = [
    "ListingKey", "CloseDate", "ListingContractDate", "PurchaseContractDate",
    "ClosePrice", "ListPrice", "LivingArea", "DaysOnMarket",
    "BedroomsTotal", "BathroomsTotalInteger", "PropertySubType",
    "CountyOrParish", "City", "StateOrProvince", "PostalCode",
    "Latitude", "Longitude", "price_per_sqft",
]
review_columns = [column for column in review_columns if column in investigation.columns]

SAMPLE_SIZE = 20

for flag in flag_columns:
    mask = investigation[flag].fillna(False).astype(bool)
    flagged = investigation.loc[mask, review_columns + [flag]]
    print(f"\n{flag}: {len(flagged):,} rows")
    display(flagged.head(SAMPLE_SIZE))


close_price_outlier_flag: 4,421 rows


,ListingKey,CloseDate,ListingContractDate,PurchaseContractDate,ClosePrice,ListPrice,LivingArea,DaysOnMarket,BedroomsTotal,BathroomsTotalInteger,PropertySubType,CountyOrParish,City,StateOrProvince,PostalCode,Latitude,Longitude,price_per_sqft,close_price_outlier_flag
23,1059617256,2024-01-31,2023-11-17,2023-12-12,9635000.0,10750000.0,5410.0,25,6.0,8.0,SingleFamilyResidence,Los Angeles,Beverly Hills,CA,90210,34.082865,-118.398812,1780.961183,True
24,1059616313,2024-01-31,2024-01-31,2024-01-31,8720000.0,8795000.0,6424.0,0,5.0,6.0,SingleFamilyResidence,Los Angeles,Los Angeles,CA,90049,34.073665,-118.488375,1357.409714,True
39,1059550612,2024-01-30,2024-01-30,2024-01-30,7000000.0,7000000.0,7500.0,0,5.0,6.0,SingleFamilyResidence,Ventura,Westlake Village,CA,91362,34.192576,-118.801414,933.333333,True
44,1059542455,2024-01-30,2024-01-30,2024-01-30,6620000.0,6620000.0,2888.0,0,3.0,3.0,SingleFamilyResidence,Orange,Newport Beach,CA,92663,33.629150,-117.955585,2292.243767,True
82,1059356812,2024-01-29,2024-01-26,2024-01-26,7600000.0,7500000.0,4482.0,0,4.0,5.0,SingleFamilyResidence,Santa Clara,Los Altos Hills,CA,94022,37.380363,-122.143880,1695.671575,True
106,1058466473,2024-01-22,2023-09-17,2024-01-19,8325000.0,8995000.0,4610.0,81,4.0,5.0,SingleFamilyResidence,San Mateo,Hillsborough,CA,94010,37.569633,-122.353623,1805.856833,True
108,1058465848,2024-01-24,2024-01-22,2024-01-22,6700000.0,6950000.0,5524.0,0,4.0,5.0,SingleFamilyResidence,Santa Clara,Saratoga,CA,95070,37.244358,-122.021385,1212.889211,True
162,1058391576,2024-01-18,2024-01-18,2024-01-18,7950000.0,7950000.0,NaN,0,4.0,5.0,SingleFamilyResidence,Los Angeles,Santa Monica,CA,90402,34.041233,-118.492371,NaN,True
204,1058224368,2024-01-24,2023-11-21,2024-01-02,13850000.0,11026000.0,14543.0,42,9.0,13.0,SingleFamilyResidence,Los Angeles,Los Angeles,CA,90049,34.075270,-118.459782,952.348209,True
267,1055495035,2024-01-08,2024-01-04,2024-01-04,6117000.0,6117000.0,5439.0,0,5.0,6.0,SingleFamilyResidence,Los Angeles,Los Angeles,CA,90049,34.078666,-118.490683,1124.655268,True



implausible_coordinate_flag: 85 rows


,ListingKey,CloseDate,ListingContractDate,PurchaseContractDate,ClosePrice,ListPrice,LivingArea,DaysOnMarket,BedroomsTotal,BathroomsTotalInteger,PropertySubType,CountyOrParish,City,StateOrProvince,PostalCode,Latitude,Longitude,price_per_sqft,implausible_coordinate_flag
9713,1044978778,2024-01-23,2023-09-08,2023-11-08,735000.0,775000.0,1092.0,59,3.0,1.0,SingleFamilyResidence,Orange,Santa Ana,CA,92703,1.000000,2.000000,673.076923,True
23076,1045537743,2024-02-02,2023-09-17,2023-12-01,730000.0,729000.0,1724.0,71,5.0,3.0,Duplex,Riverside,Riverside,CA,92504,33.934625,117.404219,423.433875,True
27959,1061262345,2024-03-26,2024-02-09,2024-02-29,385000.0,405000.0,946.0,14,2.0,2.0,SingleFamilyResidence,San Bernardino,Sugarloaf,CA,92386,48.428421,-123.365644,406.976744,True
37264,1050732402,2024-03-04,2023-12-03,2023-12-16,710080.0,709655.0,1432.0,13,3.0,3.0,Townhouse,San Diego,Spring Valley,CA,91977,32.726310,116.973758,495.865922,True
62737,1068094062,2024-05-09,2024-04-12,2024-04-12,315000.0,315000.0,892.0,0,2.0,1.0,SingleFamilyResidence,Merced,Merced,CA,95340,50.000000,100.000000,353.139013,True
65551,1065274412,2024-05-13,2024-04-04,2024-04-15,425000.0,425000.0,1673.0,11,3.0,2.0,SingleFamilyResidence,San Bernardino,Victorville,CA,92392,34.475681,117.372126,254.034668,True
67735,1064940544,2024-05-02,2024-03-27,2024-04-02,805000.0,771000.0,1822.0,6,3.0,3.0,SingleFamilyResidence,Los Angeles,Carson,CA,90746,33.852530,118.251830,441.822173,True
74984,1048467737,2024-05-14,2023-11-15,2023-12-12,1994637.0,1799000.0,2922.0,25,4.0,5.0,SingleFamilyResidence,San Luis Obispo,San Luis Obispo,CA,93401,35.251598,120.634389,682.627310,True
75159,1046655870,2024-05-13,2023-11-01,2024-05-02,3650000.0,3999000.0,2506.0,183,4.0,4.0,SingleFamilyResidence,Monterey,Carmel,CA,93923,10.305220,-84.810691,1456.504389,True
88668,1065220693,2024-06-04,2024-03-27,2024-04-30,1250000.0,1198000.0,1624.0,22,3.0,2.0,Condominium,Orange,Laguna Woods,CA,92637,-38.718318,-62.266348,769.704433,True



invalid_close_price_flag: 1 rows


...

,ListingKey,CloseDate,ListingContractDate,PurchaseContractDate,ClosePrice,ListPrice,LivingArea,DaysOnMarket,BedroomsTotal,BathroomsTotalInteger,PropertySubType,CountyOrParish,City,StateOrProvince,PostalCode,Latitude,Longitude,price_per_sqft,price_per_sqft_outlier_flag
44,1059542455,2024-01-30,2024-01-30,2024-01-30,6620000.0,6620000.0,2888.0,0,3.0,3.0,SingleFamilyResidence,Orange,Newport Beach,CA,92663,33.629150,-117.955585,2292.243767,True
120,1058424214,2024-01-19,2024-01-19,2024-01-19,2400000.0,2400000.0,979.0,0,2.0,1.0,SingleFamilyResidence,Santa Clara,Palo Alto,CA,94301,37.441520,-122.152688,2451.481103,True
216,1058209002,2024-01-23,2024-01-09,2024-01-18,3620000.0,3595000.0,1600.0,9,3.0,2.0,SingleFamilyResidence,Santa Barbara,Montecito,CA,93108,34.422741,-119.627983,2262.500000,True
325,1054780486,2024-01-31,2024-01-10,2024-01-18,7500000.0,7200000.0,2900.0,8,4.0,4.0,SingleFamilyResidence,Santa Clara,Palo Alto,CA,94301,37.434626,-122.141305,2586.206897,True
331,1054718190,2024-01-10,2024-01-10,2024-01-10,9900000.0,9900000.0,2131.0,0,4.0,3.0,SingleFamilyResidence,Santa Clara,Los Altos Hills,CA,94022,37.382339,-122.163266,4645.706241,True
381,1054211833,2024-01-02,2023-11-02,2023-11-02,18500000.0,18500000.0,2377.0,0,3.0,5.0,SingleFamilyResidence,San Diego,Del Mar,CA,92014,32.966123,-117.268304,7782.919647,True
511,1054132579,2024-01-12,2024-01-05,2024-01-05,3150000.0,2995000.0,1335.0,0,3.0,2.0,SingleFamilyResidence,Santa Clara,Palo Alto,CA,94303,37.432196,-122.118958,2359.550562,True
1489,1052411536,2024-01-11,2023-12-15,2023-12-21,2480000.0,2200000.0,1280.0,6,3.0,2.0,SingleFamilyResidence,Santa Clara,Mountain View,CA,94040,37.373138,-122.092591,1937.500000,True
1504,1052409484,2024-01-22,2023-12-15,2023-12-20,2300000.0,2198000.0,974.0,5,3.0,1.0,SingleFamilyResidence,Santa Clara,Mountain View,CA,94040,37.390239,-122.097370,2361.396304,True
1593,1052388019,2024-01-24,2023-12-14,2023-12-22,6225000.0,6495000.0,2555.0,7,4.0,3.0,SingleFamilyResidence,San Diego,San Diego,CA,92109,32.773807,-117.252890,2436.399217,True



purchase_after_close_flag: 240 rows


,ListingKey,CloseDate,ListingContractDate,PurchaseContractDate,ClosePrice,ListPrice,LivingArea,DaysOnMarket,BedroomsTotal,BathroomsTotalInteger,PropertySubType,CountyOrParish,City,StateOrProvince,PostalCode,Latitude,Longitude,price_per_sqft,purchase_after_close_flag
5,1061988701,2024-01-31,2024-01-31,2024-03-04,2340000.0,2340000.0,2442.0,0,4.0,3.0,SingleFamilyResidence,Santa Clara,San Jose,CA,95148,37.325454,-121.780303,958.230958,True
9,1060105211,2024-01-31,2024-01-31,2024-02-05,2150000.0,2150000.0,2200.0,0,3.0,2.0,SingleFamilyResidence,San Mateo,Millbrae,CA,94030,37.589762,-122.399887,977.272727,True
10,1060066702,2024-01-29,2024-01-29,2024-02-04,2000000.0,2000000.0,1690.0,0,3.0,2.0,SingleFamilyResidence,San Mateo,San Mateo,CA,94403,37.535899,-122.282826,1183.431953,True
18,1059861172,2024-01-29,2024-01-29,2024-02-01,1928800.0,1928800.0,3799.0,0,4.0,5.0,SingleFamilyResidence,Alameda,Hayward,CA,94542,37.657417,-122.013131,507.712556,True
80,1059357902,2024-01-25,2024-01-25,2024-01-26,1100000.0,1100000.0,892.0,0,2.0,1.0,SingleFamilyResidence,Santa Cruz,Santa Cruz,CA,95060,36.957622,-122.045778,1233.183857,True
115,1058442145,2024-01-10,2024-01-10,2024-01-21,2950000.0,2950000.0,2272.0,0,4.0,3.0,SingleFamilyResidence,Santa Cruz,Watsonville,CA,95076,36.855707,-121.812607,1298.415493,True
121,1058422805,2024-01-16,2024-01-16,2024-01-19,2000000.0,2000000.0,1590.0,0,3.0,2.0,SingleFamilyResidence,San Mateo,Burlingame,CA,94010,37.589295,-122.380328,1257.861635,True
165,1058387951,2024-01-16,2024-01-16,2024-01-18,2155000.0,2155000.0,2176.0,0,4.0,3.0,SingleFamilyResidence,Alameda,Fremont,CA,94555,37.562665,-122.063527,990.349265,True
246,1058155157,2024-01-13,2024-01-13,2024-01-14,899000.0,899000.0,1506.0,0,2.0,2.0,SingleFamilyResidence,Los Angeles,Rosemead,CA,91770,34.046497,-118.084969,596.945551,True
11196,1071117377,2024-02-21,2024-02-21,2024-04-22,1520000.0,1520000.0,1892.0,0,3.0,4.0,Townhouse,Santa Clara,Santa Clara,CA,95050,NaN,NaN,803.382664,True



zero_coordinate_flag: 37 rows


,ListingKey,CloseDate,ListingContractDate,PurchaseContractDate,ClosePrice,ListPrice,LivingArea,DaysOnMarket,BedroomsTotal,BathroomsTotalInteger,PropertySubType,CountyOrParish,City,StateOrProvince,PostalCode,Latitude,Longitude,price_per_sqft,zero_coordinate_flag
137991,1076725468,2024-09-12,2024-06-20,2024-08-31,420000.0,434900.0,1100.0,71,3.0,2.0,SingleFamilyResidence,Los Angeles,Lancaster,CA,93535,0.0,0.0,381.818182,True
141060,1064843666,2024-09-06,2024-03-23,2024-04-25,360000.0,399000.0,800.0,31,2.0,2.0,Duplex,San Bernardino,San Bernardino,CA,92410,0.0,0.0,450.000000,True
157171,1072840823,2024-10-16,2024-04-25,2024-05-20,1060487.0,1019057.0,3732.0,25,6.0,4.0,SingleFamilyResidence,San Joaquin,Manteca,CA,95337,0.0,0.0,284.160504,True
158224,1093509478,2024-11-22,2024-11-01,2024-11-15,844000.0,874102.0,2202.0,78,4.0,3.0,SingleFamilyResidence,San Benito,Hollister,CA,95023,0.0,0.0,383.287920,True
180157,1089715991,2024-12-23,2024-10-06,2024-11-16,1528000.0,1548000.0,2176.0,41,4.0,4.0,SingleFamilyResidence,Contra Costa,San Ramon,CA,94583,0.0,0.0,702.205882,True
183308,1082151797,2024-12-02,2024-08-31,2024-11-06,1638000.0,1638000.0,2219.0,72,4.0,5.0,SingleFamilyResidence,Contra Costa,San Ramon,CA,94583,0.0,0.0,738.170347,True
184472,1079170121,2024-12-26,2024-08-05,2024-10-25,999097.0,999097.0,1892.0,81,4.0,3.0,SingleFamilyResidence,Santa Clara,Gilroy,CA,95020,0.0,0.0,528.063953,True
192014,1093721468,2025-01-07,2024-11-06,2024-12-10,1060000.0,1049000.0,1939.0,34,4.0,3.0,Condominium,San Diego,San Diego,CA,92128,0.0,0.0,546.673543,True
193883,1089985539,2025-01-27,2024-10-11,2024-10-24,1206569.0,1218660.0,2452.0,13,4.0,3.0,SingleFamilyResidence,Santa Clara,Gilroy,CA,95020,0.0,0.0,492.075449,True
196613,1077164887,2025-01-22,2024-07-06,2024-09-07,1016386.0,1002149.0,2008.0,63,4.0,3.0,SingleFamilyResidence,Santa Clara,Gilroy,CA,95020,0.0,0.0,506.168327,True


## 3. Invalid Numeric Values

- `ClosePrice <= 0`: consider removing the transaction because close price is essential to sold-market analysis.
- `LivingArea <= 0`: retain the transaction and replace the invalid area with missing.
- `DaysOnMarket < 0`: retain the transaction and replace the invalid DOM with missing.
- Negative bedroom or bathroom values: retain the transaction and replace the invalid field with missing.

In [26]:
numeric_flags = [
    "invalid_close_price_flag", "invalid_living_area_flag",
    "negative_days_on_market_flag", "negative_bedrooms_flag",
    "negative_bathrooms_flag",
]
numeric_flags = [flag for flag in numeric_flags if flag in investigation.columns]
numeric_columns = [
    "ListingKey", "ClosePrice", "LivingArea", "DaysOnMarket",
    "BedroomsTotal", "BathroomsTotalInteger",
] + numeric_flags
display(investigation.loc[investigation[numeric_flags].any(axis=1), numeric_columns])

,ListingKey,ClosePrice,LivingArea,DaysOnMarket,BedroomsTotal,BathroomsTotalInteger,invalid_close_price_flag,invalid_living_area_flag,negative_days_on_market_flag,negative_bedrooms_flag,negative_bathrooms_flag
4,1063453216,1950000.0,2100.0,-36,3.0,0.0,False,False,True,False,False
393,1054197723,655000.0,3045.0,-1,5.0,4.0,False,False,True,False,False
7701,1046380894,8000000.0,0.0,62,4.0,5.0,False,True,False,False,False
11204,1062122119,899000.0,2533.0,-13,2.0,3.0,False,False,True,False,False
11371,1061302766,372000.0,1488.0,-10,3.0,2.0,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...
445624,1153246696,700000.0,0.0,98,3.0,0.0,False,True,False,False,False
446090,1152540534,57000.0,0.0,76,0.0,0.0,False,True,False,False,False
446924,1150495367,160000.0,0.0,114,0.0,0.0,False,True,False,False,False
447179,1147474972,640000.0,0.0,153,1.0,1.0,False,True,False,False,False


## 4. Date Consistency Issues

Review listing, purchase-contract, and close dates together. Invalid date fields can be set to missing when the correct value cannot be recovered; the transaction may still be useful for non-timeline analysis.

In [27]:
date_flags = [
    "listing_after_close_flag", "purchase_after_close_flag",
    "negative_timeline_flag",
]
date_flags = [flag for flag in date_flags if flag in investigation.columns]
date_columns = [
    "ListingKey", "ListingContractDate", "PurchaseContractDate", "CloseDate",
] + date_flags
display(investigation.loc[investigation[date_flags].any(axis=1), date_columns].head(100))

,ListingKey,ListingContractDate,PurchaseContractDate,CloseDate,listing_after_close_flag,purchase_after_close_flag,negative_timeline_flag
5,1061988701,2024-01-31,2024-03-04,2024-01-31,False,True,True
9,1060105211,2024-01-31,2024-02-05,2024-01-31,False,True,True
10,1060066702,2024-01-29,2024-02-04,2024-01-29,False,True,True
18,1059861172,2024-01-29,2024-02-01,2024-01-29,False,True,True
20,1059827487,2024-01-31,2024-01-15,2024-01-30,True,False,True
...,...,...,...,...,...,...,...
54981,1054116549,2024-01-04,2024-04-26,2024-04-24,False,True,True
57335,1076006049,2024-05-31,2024-06-01,2024-05-31,False,True,True
57338,1075988530,2024-05-28,2024-05-31,2024-05-28,False,True,True
57372,1075920368,2024-05-28,2024-05-30,2024-05-28,False,True,True


## 5. Geographic Quality Issues

Missing or invalid coordinates should be excluded from maps and coordinate-based analysis. The transaction can remain if its other market fields are valid.

In [28]:
geo_flags = [
    "missing_coordinate_flag", "zero_coordinate_flag",
    "positive_longitude_flag", "implausible_coordinate_flag",
]
geo_flags = [flag for flag in geo_flags if flag in investigation.columns]
geo_columns = [
    "ListingKey", "CountyOrParish", "City", "StateOrProvince",
    "PostalCode", "Latitude", "Longitude",
] + geo_flags
display(investigation.loc[investigation[geo_flags].any(axis=1), geo_columns].head(100))

,ListingKey,CountyOrParish,City,StateOrProvince,PostalCode,Latitude,Longitude,missing_coordinate_flag,zero_coordinate_flag,positive_longitude_flag,implausible_coordinate_flag
448,1054150881,Alameda,Newark,CA,94560,NaN,NaN,True,False,False,False
671,1054077195,San Diego,San Diego,CA,92126,NaN,NaN,True,False,False,False
855,1054007898,Contra Costa,Concord,CA,94521,NaN,NaN,True,False,False,False
3566,1050249809,Alameda,San Leandro,CA,94578,NaN,NaN,True,False,False,False
3726,1049915635,Alameda,Berkeley,CA,94705,NaN,NaN,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...
11369,1061306707,Santa Cruz,Watsonville,CA,95076,NaN,NaN,True,False,False,False
11376,1061289700,Santa Clara,Santa Clara,CA,95051,NaN,NaN,True,False,False,False
11382,1061274790,San Mateo,San Mateo,CA,94402,NaN,NaN,True,False,False,False
11386,1061272000,Alameda,Oakland,CA,94608,NaN,NaN,True,False,False,False


## 6. Extreme Outliers

Outliers are retained for review because they may represent valid luxury, multi-unit, portfolio, or unusual residential transactions. Summaries and samples are more useful than manually reviewing thousands of rows.

In [29]:
outlier_flags = [
    "close_price_outlier_flag", "living_area_outlier_flag",
    "price_per_sqft_outlier_flag", "possible_non_standard_residential_flag",
]
outlier_flags = [flag for flag in outlier_flags if flag in investigation.columns]
outlier_columns = [
    "ListingKey", "ClosePrice", "LivingArea", "price_per_sqft",
    "PropertySubType", "CountyOrParish", "City",
] + outlier_flags

display(investigation[["ClosePrice", "LivingArea", "price_per_sqft"]].describe(
    percentiles=[0.5, 0.9, 0.95, 0.99, 0.995]
).T)

display(
    investigation.loc[investigation[outlier_flags].any(axis=1), outlier_columns]
    .sort_values("ClosePrice", ascending=False)
    .head(100)
)

,count,mean,std,min,50%,90%,95%,99%,99.5%,max
ClosePrice,447960.0,1.192668e+06,6.058565e+06,0.0,825000.000000,2.075000e+06,2.858000e+06,5.600000e+06,7.550000e+06,9.895000e+08
LivingArea,447711.0,1.904286e+03,2.545591e+04,0.0,1646.000000,2.984000e+03,3.564000e+03,5.288000e+03,6.280000e+03,1.702132e+07
price_per_sqft,447544.0,6.462383e+02,5.047426e+03,0.0,537.294564,9.913793e+02,1.217712e+03,1.895037e+03,2.233660e+03,1.164067e+06


,ListingKey,ClosePrice,LivingArea,price_per_sqft,PropertySubType,CountyOrParish,City,close_price_outlier_flag,living_area_outlier_flag,price_per_sqft_outlier_flag,possible_non_standard_residential_flag
319715,1133720285,989500000.0,2289.0,432284.840542,SingleFamilyResidence,Los Angeles,Valencia,True,False,True,True
237063,1100948167,970000000.0,1976.0,490890.688259,SingleFamilyResidence,San Diego,Chula Vista,True,False,True,True
270346,1076630163,900000000.0,11000.0,81818.181818,SingleFamilyResidence,Kern,Glennville,True,True,True,True
310591,1118793238,890000000.0,1444.0,616343.490305,Condominium,San Diego,San Diego,True,False,True,True
301507,1136956787,875000000.0,1730.0,505780.346821,SingleFamilyResidence,San Diego,San Diego,True,False,True,True
...,...,...,...,...,...,...,...,...,...,...,...
430678,1174662172,31250000.0,6742.0,4635.123109,SingleFamilyResidence,Los Angeles,Malibu,True,True,True,True
3263,1050717196,31000000.0,8140.0,3808.353808,SingleFamilyResidence,Orange,Laguna Beach,True,True,True,True
416330,1158076210,31000000.0,12652.0,2450.205501,SingleFamilyResidence,Los Angeles,Los Angeles,True,True,True,True
270553,1119600136,30500000.0,11154.0,2734.445042,SingleFamilyResidence,Los Angeles,Malibu,True,True,True,True
